In [1]:
# ============================================================
#  PASSWORD STRENGTH ANALYZER & SECURITY SUGGESTION TOOL
#  Single-cell Colab build. Defensive, local, privacy-first.
#  Never stores/logs passwords. Use ONLY synthetic passwords in demos.
# ============================================================
import subprocess, sys, os, re, math, json, time, sqlite3, secrets, string, hashlib, hmac, threading, logging, random
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flask"], check=True)
from flask import Flask, request, jsonify, Response
from IPython.display import display, HTML

# ---------------- DATA (small educational list, no leaked credentials) ----------------
os.makedirs("data", exist_ok=True)
_common = """password password1 password123 123456 1234567 12345678 123456789 qwerty qwerty123 abc123 letmein
welcome admin admin123 iloveyou monkey dragon login master sunshine football princess hello hello123 test test123
user secret changeme default 111111 000000 1q2w3e4r 1qaz2wsx""".split()
open("data/common_passwords.txt", "w").write("\n".join(_common))
COMMON = set(open("data/common_passwords.txt").read().split())
WORDS = COMMON | set("summer winter spring autumn india cricket google apple banana cookie flower monday friday love baby shadow soccer hunter ranger thunder".split())
LEET = str.maketrans({'@': 'a', '$': 's', '3': 'e', '0': 'o', '1': 'i', '!': 'i', '5': 's', '7': 't', '4': 'a'})
ROWS = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
EXTRA_KB = ["1qaz", "2wsx", "qazwsx", "zaq1"]
CFG = dict(min_length=12, max_length=128, common_check=True, personal_check=True)   # admin-configurable policy

# ---------------- ANALYSIS ENGINE ----------------
def analyze_length(p):
    n = len(p)
    band = "Very short" if n < 8 else "Short" if n < 12 else "Better length" if n < 16 else "Strong length contribution"
    return {"length": n, "band": band}

def analyze_characters(p):
    t = {"lowercase": any(c.islower() for c in p), "uppercase": any(c.isupper() for c in p),
         "digits": any(c.isdigit() for c in p), "symbols": any((not c.isalnum()) and c != " " for c in p),
         "spaces": " " in p}
    uniq = len(set(p))
    return {"types": t, "type_count": sum(t[k] for k in ("lowercase", "uppercase", "digits", "symbols")),
            "unique_count": uniq, "unique_ratio": round(uniq / max(len(p), 1), 3)}

def is_common_password(p):
    low = p.lower()
    return low in COMMON or low.translate(LEET) in COMMON

def detect_sequences(p, min_len=4):
    s, out, i = p.lower(), [], 0
    while i < len(s) - 1:
        a, b = s[i], s[i + 1]
        d = ord(b) - ord(a)
        if d in (1, -1) and a.isalnum() and b.isalnum() and a.isdigit() == b.isdigit():
            j = i + 1
            while j + 1 < len(s) and ord(s[j + 1]) - ord(s[j]) == d and s[j + 1].isalnum() and s[j + 1].isdigit() == s[j].isdigit():
                j += 1
            if j - i + 1 >= min_len: out.append("ascending" if d == 1 else "descending")
            i = j
        else:
            i += 1
    return out

def detect_keyboard_patterns(p, w=4):
    s, hits = p.lower(), set()
    for row in ROWS:
        for i in range(len(row) - w + 1):
            seg = row[i:i + w]
            if seg in s or seg[::-1] in s: hits.add(seg)
    hits |= {k for k in EXTRA_KB if k in s}
    return sorted(hits)

def detect_repetition(p):
    f = []
    if re.search(r"(.)\1{3,}", p): f.append("repeated character")
    if re.search(r"(.{2,4})\1{2,}", p): f.append("repeated substring")
    return f

def detect_predictable_structure(p):
    low = p.lower()
    base = re.sub(r"[\d!@#$%^&*._\-]+$", "", low)
    norm = base.translate(LEET)
    res = {"common_base": False, "word_base": False, "year": bool(re.search(r"(19|20)\d{2}", p))}
    if norm and (len(base) != len(low) or norm != base) and norm != low:
        res["common_base"] = norm in COMMON
        res["word_base"] = norm in WORDS
    return res

def check_context(p, ctx):
    low, norm, hits = p.lower(), p.lower().translate(LEET), []
    for k, v in (ctx or {}).items():
        v = (v or "").strip().lower()
        if (len(v) >= 3 or (k == "year" and len(v) == 4)) and (v in low or v in norm): hits.append(k)
    return hits

def estimate_theoretical_entropy(p):
    pool = (26 if any(c.islower() for c in p) else 0) + (26 if any(c.isupper() for c in p) else 0) + \
           (10 if any(c.isdigit() for c in p) else 0) + (33 if any((not c.isalnum()) for c in p) else 0)
    return round(len(p) * math.log2(max(pool, 1)), 1), pool   # H ~ L*log2(N): assumes RANDOM choice

def guess_space_label(bits):   # educational only, NOT a time-to-crack claim
    return "very small" if bits < 28 else "small" if bits < 45 else "moderate" if bits < 60 else "large" if bits < 80 else "very large"

def classify(s):
    return "VERY WEAK" if s <= 20 else "WEAK" if s <= 40 else "MODERATE" if s <= 60 else "STRONG" if s <= 80 else "VERY STRONG"

def check_policy(p, c, common, ctx_hits, cfg=CFG):
    r = [("Minimum length %d" % cfg["min_length"], len(p) >= cfg["min_length"]),
         ("Within max length %d" % cfg["max_length"], len(p) <= cfg["max_length"])]
    if cfg["common_check"]: r.append(("Not a common password", not common))
    if cfg["personal_check"]: r.append(("No personal info", not ctx_hits))
    return {"status": "POLICY PASS" if all(x[1] for x in r) else "POLICY FAIL", "rules": [{"rule": a, "ok": b} for a, b in r]}

SUGGEST = {
    "common": "This matches a commonly used password and should not be used anywhere.",
    "sequence": "A predictable numeric/letter sequence was detected. Break it up or avoid it.",
    "keyboard": "A keyboard pattern (like qwerty or asdf) was detected. Keyboard walks are among the first guesses.",
    "repetition": "Repeated characters or chunks reduce unpredictability. Avoid repeating patterns.",
    "structure": "A common word with a number/symbol tail is predictable. Adding digits to a known word adds little strength.",
    "year": "A year-like number was detected. Avoid birth years and current years.",
    "personal": "Your password appears to include personal information. Avoid names, birth years, and college/company names.",
    "short": "Consider a longer password or a passphrase of 4+ random words (16+ characters).",
}
GENERAL = ["Use a password manager to generate and store a unique password for every account.",
           "Never reuse passwords across accounts; reuse enables credential stuffing.", "Enable MFA wherever available."]

def generate_suggestions(findings, length):
    types = {f["type"] for f in findings}
    out = [SUGGEST[t] for t in ("common", "sequence", "keyboard", "repetition", "structure", "year", "personal") if t in types]
    if length < 16: out.append(SUGGEST["short"])
    return out + GENERAL

def analyze_password(p, ctx=None):
    n = len(p)
    if n == 0:
        return {"score": 0, "classification": "VERY WEAK", "findings": [], "strengths": [], "suggestions": ["Start typing to see analysis."],
                "metrics": {"length": 0}, "policy": check_policy("", {}, False, []), "pattern_count": 0}
    L, C = analyze_length(p), analyze_characters(p)
    common, seqs, kb, rep = is_common_password(p), detect_sequences(p), detect_keyboard_patterns(p), detect_repetition(p)
    st, hits = detect_predictable_structure(p), check_context(p, ctx)
    bits, pool = estimate_theoretical_entropy(p)
    F, S = [], []
    def add(t, sev, d): F.append({"type": t, "severity": sev, "description": d})
    if common: add("common", "high", "Matches a commonly used password pattern.")
    if seqs: add("sequence", "medium", "Predictable %s sequence detected." % "/".join(sorted(set(seqs))))
    if kb: add("keyboard", "medium", "Keyboard pattern detected (%s)." % ", ".join(kb))
    if rep: add("repetition", "medium", "Repetition detected (%s)." % ", ".join(rep))
    if st["common_base"] or st["word_base"]: add("structure", "high" if st["common_base"] else "medium", "Common word plus predictable number/symbol tail.")
    if st["year"]: add("year", "low", "Year/date-like number detected.")
    if hits: add("personal", "high", "Password appears to contain personal information (%s)." % ", ".join(hits))
    if n < 12: add("short", "medium", "Length is %s (%d characters)." % (L["band"].lower(), n))
    if n >= 12: S.append("Good length (%d characters)" % n)
    if C["type_count"] >= 3: S.append("Good character variety")
    if C["unique_ratio"] >= 0.7: S.append("High unique-character ratio")
    if not (common or seqs or kb or rep): S.append("No common/sequence/keyboard/repeat patterns")
    pat_types = {"sequence", "keyboard", "repetition", "structure", "personal"}
    pcount = len([f for f in F if f["type"] in pat_types])
    # --- scoring: positives ---
    score = min(35, n * 35 / 16) + C["type_count"] * 3.75 + C["unique_ratio"] * 10
    score += max(0, 20 - 7 * pcount) + (0 if common else 10)
    score += min(10, bits / 10) if pcount == 0 and not common else 0
    # --- penalties ---
    score -= 50 * common + 12 * bool(seqs) + 15 * bool(kb) + 15 * bool(rep) + 20 * bool(hits) + 15 * (st["common_base"] or st["word_base"])
    # --- transparent caps ---
    cap = 100
    if common: cap = min(cap, 15)
    if st["common_base"]: cap = min(cap, 40)
    elif st["word_base"]: cap = min(cap, 55)
    if C["unique_ratio"] < 0.35 and n >= 6: cap = min(cap, 30)
    if hits: cap = min(cap, 50)
    score = int(max(0, min(cap, round(score))))
    eff = bits if pcount == 0 and not common else bits * 0.5
    return {"score": score, "classification": classify(score), "findings": F, "strengths": S,
            "suggestions": generate_suggestions(F, n), "pattern_count": pcount,
            "policy": check_policy(p, C, common, hits),
            "metrics": {"length": n, "length_band": L["band"], "type_count": C["type_count"], "unique_count": C["unique_count"],
                        "unique_ratio": C["unique_ratio"], "theoretical_entropy_bits": bits, "char_pool": pool,
                        "guess_space_label": guess_space_label(eff), "note": "Educational estimate only."}}

# ---------------- SECURE GENERATOR (secrets, not random) ----------------
def generate_password(length=20, up=True, low=True, num=True, sym=True):
    pools = [s for s, on in ((string.ascii_uppercase, up), (string.ascii_lowercase, low), (string.digits, num), ("!@#$%^&*()-_=+[]{}:,.?", sym)) if on] or [string.ascii_lowercase]
    chars = [secrets.choice(p) for p in pools] + [secrets.choice("".join(pools)) for _ in range(length - len(pools))]
    random.SystemRandom().shuffle(chars)   # OS-level CSPRNG
    return "".join(chars)

# ---------------- EDUCATIONAL HASHING DEMO (separate from analyzer) ----------------
def hash_password(pw):
    salt = os.urandom(16)
    h = hashlib.scrypt(pw.encode(), salt=salt, n=2**14, r=8, p=1, dklen=32)
    return "scrypt$%s$%s" % (salt.hex(), h.hex())

def verify_password(pw, stored):
    _, salt, h = stored.split("$")
    cand = hashlib.scrypt(pw.encode(), salt=bytes.fromhex(salt), n=2**14, r=8, p=1, dklen=32)
    return hmac.compare_digest(cand.hex(), h)

# ---------------- ANALYTICS DB (metadata only, NO password column) ----------------
DB, LOCK = "analytics.db", threading.Lock()
def db():
    c = sqlite3.connect(DB); c.row_factory = sqlite3.Row; return c
with db() as c:
    c.executescript("""CREATE TABLE IF NOT EXISTS analyses(analysis_id INTEGER PRIMARY KEY AUTOINCREMENT, score INTEGER, classification TEXT,
      password_length INTEGER, unique_character_ratio REAL, weakness_count INTEGER, pattern_count INTEGER, created_at TEXT);
      CREATE TABLE IF NOT EXISTS findings(finding_id INTEGER PRIMARY KEY AUTOINCREMENT, analysis_id INTEGER, finding_type TEXT, severity TEXT, description TEXT);""")

def record(res):   # takes the RESULT only; the password never reaches this function
    with LOCK, db() as c:
        cur = c.execute("INSERT INTO analyses(score,classification,password_length,unique_character_ratio,weakness_count,pattern_count,created_at) VALUES(?,?,?,?,?,?,datetime('now'))",
                        (res["score"], res["classification"], res["metrics"].get("length", 0), res["metrics"].get("unique_ratio", 0), len(res["findings"]), res["pattern_count"]))
        for f in res["findings"]:
            c.execute("INSERT INTO findings(analysis_id,finding_type,severity,description) VALUES(?,?,?,?)", (cur.lastrowid, f["type"], f["severity"], f["description"]))

def stats():
    with db() as c:
        rows = c.execute("SELECT * FROM analyses").fetchall()
        cls = {k: 0 for k in ["VERY WEAK", "WEAK", "MODERATE", "STRONG", "VERY STRONG"]}
        for r in rows: cls[r["classification"]] += 1
        hist, lens = [0] * 10, {"<8": 0, "8-11": 0, "12-15": 0, "16-23": 0, "24+": 0}
        for r in rows:
            hist[min(r["score"] // 10, 9)] += 1; n = r["password_length"]
            lens["<8" if n < 8 else "8-11" if n < 12 else "12-15" if n < 16 else "16-23" if n < 24 else "24+"] += 1
        w = {r[0]: r[1] for r in c.execute("SELECT finding_type, COUNT(*) FROM findings GROUP BY 1 ORDER BY 2 DESC")}
    pats = {k: w.get(k, 0) for k in ("sequence", "keyboard", "repetition", "structure", "personal")}
    return {"total": len(rows), "classes": cls, "average": round(sum(r["score"] for r in rows) / max(len(rows), 1), 1),
            "score_hist": hist, "weaknesses": w, "lengths": lens, "patterns": pats}

# ---------------- SEED DASHBOARD WITH SYNTHETIC DEMO DATA ----------------
for demo in ["123456", "password", "Password123!", "qwerty2026!", "aaaaaaaaaaaaaaaa", "abcd1234", "welcome123", "admin2026", "letmein", "hello1234",
             "P@ssw0rd2024", "ababababab", "Summer2025!", "9876543210", "asdfghjkl", "zxcvbn99", "monkey", "iloveyou1", "Dragon#1999", "test123"] \
            + [generate_password(random.choice([16, 20, 24])) for _ in range(12)] + ["velvet-galaxy-harbor-orchid-drift", "river candle photon saffron ember"]:
    record(analyze_password(demo))

# ---------------- WEB APP ----------------
logging.getLogger("werkzeug").setLevel(logging.ERROR)
app = Flask(__name__); app.logger.disabled = True
HITS = {}
def limited():
    ip, now = request.remote_addr, time.time()
    HITS[ip] = [t for t in HITS.get(ip, []) if now - t < 60] + [now]
    return len(HITS[ip]) > 240

@app.post("/api/analyze")
def api_analyze():
    if limited(): return jsonify(error="Too many requests"), 429
    d = request.get_json(silent=True) or {}
    pw = d.get("password")
    if not isinstance(pw, str) or len(pw) > CFG["max_length"]: return jsonify(error="Invalid input"), 400   # never echo the input
    res = analyze_password(pw, d.get("context"))
    if d.get("record"): record(res)
    return jsonify(res)

@app.post("/api/generate-password")
def api_gen():
    d = request.get_json(silent=True) or {}
    ln = d.get("length", 20); ln = ln if ln in (16, 20, 24) else 20
    return jsonify(password=generate_password(ln, d.get("upper", True), d.get("lower", True), d.get("numbers", True), d.get("symbols", True)))

@app.get("/api/dashboard/stats")
def api_stats(): return jsonify(stats())

@app.get("/api/analytics/weaknesses")
def api_weak(): return jsonify(stats()["weaknesses"])

CSS = """<style>
:root{--bg:#0a0f24;--card:#121a3a;--card2:#18224b;--line:#2a3670;--tx:#f1f4ff;--mu:#aab5de;--ac:#8b9bff;--g:#2ee6a6;--y:#ffd166;--o:#ff9f43;--r:#ff5d73;--b:#4cc9f0}
*{box-sizing:border-box}body{margin:0;color:var(--tx);font-family:Inter,system-ui,'Segoe UI',sans-serif;min-height:100vh;
background:radial-gradient(1100px 600px at 8% -10%,#1d2a6e 0,transparent 60%),radial-gradient(900px 500px at 100% 0,#0d4057 0,transparent 55%),var(--bg)}
nav{display:flex;justify-content:space-between;align-items:center;padding:18px 32px;border-bottom:1px solid var(--line);backdrop-filter:blur(8px)}
.logo{font-weight:800;font-size:18px;letter-spacing:.3px}.logo span{color:var(--g)}nav a{color:var(--mu);text-decoration:none;margin-left:22px;font-weight:600}nav a.on,nav a:hover{color:#fff}
main{max-width:1180px;margin:0 auto;padding:28px}.card{background:linear-gradient(160deg,var(--card),var(--card2));border:1px solid var(--line);border-radius:18px;padding:22px;box-shadow:0 12px 40px rgba(0,0,0,.35)}
h1{font-size:30px;margin:0 0 6px}h2{font-size:16px;margin:0 0 14px;color:#fff}p.sub{color:var(--mu);margin:0 0 22px}.grid{display:grid;gap:18px}
input[type=password],input[type=text]{width:100%;padding:14px 16px;border-radius:12px;border:1px solid var(--line);background:#0c1330;color:#fff;font-size:17px;outline:none}
input:focus{border-color:var(--ac);box-shadow:0 0 0 3px rgba(139,155,255,.25)}label{color:var(--mu);font-size:13px;font-weight:600}
button{background:var(--ac);color:#0a0f24;border:0;border-radius:10px;padding:10px 16px;font-weight:700;cursor:pointer}button.ghost{background:transparent;color:#fff;border:1px solid var(--line)}button:hover{filter:brightness(1.1)}
.bar{height:14px;background:#0c1330;border-radius:99px;overflow:hidden;border:1px solid var(--line)}.bar i{display:block;height:100%;width:0;border-radius:99px;transition:all .35s}
.pill{display:inline-block;padding:5px 12px;border-radius:99px;font-weight:800;font-size:13px;color:#0a0f24}
li{margin:7px 0;line-height:1.45}ul{padding-left:20px;margin:0}.ok{color:var(--g)}.warn{color:var(--y)}.mu{color:var(--mu)}
.kpis{grid-template-columns:repeat(auto-fit,minmax(170px,1fr))}.kpi b{display:block;font-size:32px;margin-top:4px}.kpi span{color:var(--mu);font-size:13px;font-weight:600}
.two{grid-template-columns:repeat(auto-fit,minmax(340px,1fr))}.row{display:flex;gap:10px;flex-wrap:wrap;align-items:center}select{background:#0c1330;color:#fff;border:1px solid var(--line);border-radius:8px;padding:8px}
canvas{max-height:300px}.gen{font-family:ui-monospace,Menlo,monospace;background:#0c1330;border:1px dashed var(--ac);padding:12px;border-radius:10px;word-break:break-all;margin-top:12px}
</style>"""
NAV = '<nav><div class="logo">🛡️ Pass<span>Guard</span> Analyzer</div><div><a href="/" class="%s">Analyzer</a><a href="/dashboard" class="%s">Dashboard</a></div></nav>'

HOME = """<!doctype html><html><head><meta charset=utf-8><meta name=viewport content="width=device-width,initial-scale=1"><title>Password Strength Analyzer</title>__CSS__</head><body>__NAV__
<main><h1>Password Strength Analyzer</h1><p class=sub>Length + unpredictability + pattern resistance + common-password checks + context. Analysis runs in memory; nothing is stored. Use demo passwords only.</p>
<div class="grid" style="grid-template-columns:repeat(auto-fit,minmax(380px,1fr))"><div class=card>
<label>Password</label><div class=row style="flex-wrap:nowrap"><input id=pw type=password autocomplete=off placeholder="Type a demo password…"><button class=ghost id=tg>👁 Show</button></div>
<div style="margin:18px 0 8px" class=row><b id=cls style="font-size:22px">—</b><span id=pill></span><span class=mu id=sc></span></div><div class=bar><i id=fill></i></div>
<p class=mu id=met style="font-size:13px"></p>
<div class=row style="margin-top:6px"><button id=rec>＋ Add anonymous result to dashboard</button><span class=mu id=recm style="font-size:12px">Stores score/length only, never the password.</span></div>
<h2 style="margin-top:22px">Optional context (compared locally, not stored)</h2>
<div class=row><input type=text id=cn placeholder="First name" style="flex:1"><input type=text id=cy placeholder="Birth year" style="flex:1"><input type=text id=co placeholder="College/Company" style="flex:1"></div></div>
<div class=card><h2>✅ Strengths</h2><ul id=st></ul><h2 style="margin-top:18px">⚠️ Findings</h2><ul id=fd></ul><h2 style="margin-top:18px">💡 Suggestions</h2><ul id=sg></ul>
<h2 style="margin-top:18px">📋 Policy check <span id=pol class=pill></span></h2><ul id=pr></ul></div></div>
<div class="grid two" style="margin-top:18px"><div class=card><h2>🔐 Secure password generator</h2><div class=row>
<select id=gl><option>16</option><option selected>20</option><option>24</option></select><label><input type=checkbox id=gu checked> Upper</label><label><input type=checkbox id=gw checked> Lower</label>
<label><input type=checkbox id=gn checked> Numbers</label><label><input type=checkbox id=gs checked> Symbols</label><button id=gb>Generate</button></div><div class=gen id=go>Click Generate (uses Python secrets, never stored)</div></div>
<div class=card><h2>📘 Entropy &amp; passphrases</h2><p class=mu style="line-height:1.6">Entropy ≈ L × log₂(N) assumes <b>random</b> choice. Humans aren't random, so <b>Password123!</b> looks diverse but is predictable. Long, random, unique passphrases (4+ random words) are easier to remember. Guess-space labels are <b>educational estimates only</b>. Add MFA and a password manager.</p></div></div></main>
<script>const $=i=>document.getElementById(i);let t;const COL={"VERY WEAK":"#ff5d73","WEAK":"#ff9f43","MODERATE":"#ffd166","STRONG":"#2ee6a6","VERY STRONG":"#4cc9f0"};
const esc=s=>String(s).replace(/[&<>"']/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
const fill=(id,a,f)=>{$(id).innerHTML=a.length?a.map(f).join(""):"<li class=mu>None</li>"};
function body(rec){return JSON.stringify({password:$("pw").value,record:!!rec,context:{name:$("cn").value,year:$("cy").value,org:$("co").value}})}
async function run(rec){const r=await fetch("/api/analyze",{method:"POST",headers:{"Content-Type":"application/json"},body:body(rec)});if(!r.ok)return;const d=await r.json();
const c=COL[d.classification];$("cls").textContent=d.classification;$("cls").style.color=c;$("sc").textContent=d.score+"/100";$("fill").style.width=d.score+"%";$("fill").style.background=c;
const m=d.metrics;$("met").textContent=m.length?`Length ${m.length} (${m.length_band}) • ${m.type_count}/4 char types • unique ratio ${m.unique_ratio} • theoretical entropy ${m.theoretical_entropy_bits} bits • guess space: ${m.guess_space_label} (educational estimate only)`:"";
fill("st",d.strengths,x=>`<li class=ok>✓ ${esc(x)}</li>`);fill("fd",d.findings,x=>`<li class=warn>⚠ ${esc(x.description)}</li>`);fill("sg",d.suggestions,x=>`<li>${esc(x)}</li>`);
$("pol").textContent=d.policy.status;$("pol").style.background=d.policy.status=="POLICY PASS"?"#2ee6a6":"#ff5d73";fill("pr",d.policy.rules,x=>`<li class=${x.ok?"ok":"warn"}>${x.ok?"✓":"✗"} ${esc(x.rule)}</li>`);
if(rec)$("recm").textContent="Added ✓ (metadata only)"}
const go=()=>{clearTimeout(t);t=setTimeout(()=>run(false),150)};["pw","cn","cy","co"].forEach(i=>$(i).addEventListener("input",go));
$("tg").onclick=()=>{const h=$("pw").type=="password";$("pw").type=h?"text":"password";$("tg").textContent=h?"🙈 Hide":"👁 Show"};$("rec").onclick=()=>run(true);
$("gb").onclick=async()=>{const r=await fetch("/api/generate-password",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({length:+$("gl").value,upper:$("gu").checked,lower:$("gw").checked,numbers:$("gn").checked,symbols:$("gs").checked})});$("go").textContent=(await r.json()).password};</script></body></html>"""

DASH = """<!doctype html><html><head><meta charset=utf-8><meta name=viewport content="width=device-width,initial-scale=1"><title>Security Awareness Dashboard</title>__CSS__
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script></head><body>__NAV__
<main><h1>Security Awareness Dashboard</h1><p class=sub>Aggregate, privacy-safe metrics (score, length, finding types). Actual passwords are never stored or shown. Auto-refreshes every 5 s.</p>
<div class="grid kpis" id=k></div><div class="grid two" style="margin-top:18px">
<div class=card><h2>Strength distribution</h2><canvas id=c1></canvas></div><div class=card><h2>Score distribution</h2><canvas id=c2></canvas></div>
<div class=card><h2>Most common weakness types</h2><canvas id=c3></canvas></div><div class=card><h2>Password length distribution</h2><canvas id=c4></canvas></div>
<div class=card><h2>Pattern detection frequency</h2><canvas id=c5></canvas></div></div></main>
<script>Chart.defaults.color="#c9d3f5";Chart.defaults.font.family="Inter,system-ui,sans-serif";Chart.defaults.borderColor="rgba(139,155,255,.15)";
const COL=["#ff5d73","#ff9f43","#ffd166","#2ee6a6","#4cc9f0"],CH={};
function mk(id,type,labels,data,colors,opt){const o=Object.assign({plugins:{legend:{display:type=="doughnut",labels:{color:"#e8ecff"}}},scales:type=="doughnut"?{}:{x:{ticks:{color:"#c9d3f5"}},y:{ticks:{color:"#c9d3f5"},beginAtZero:true}}},opt||{});
if(CH[id]){CH[id].data.labels=labels;CH[id].data.datasets[0].data=data;CH[id].update();return}
CH[id]=new Chart($(id),{type,data:{labels,datasets:[{data,backgroundColor:colors,borderColor:"#121a3a",borderWidth:2,borderRadius:type=="bar"?8:0}]},options:o})}
const $=i=>document.getElementById(i);
async function load(){const s=await (await fetch("/api/dashboard/stats")).json();const c=s.classes;
const K=[["Total analyses",s.total,"#fff"],["Average score",s.average,"#8b9bff"],["Very weak",c["VERY WEAK"],COL[0]],["Weak",c["WEAK"],COL[1]],["Moderate",c["MODERATE"],COL[2]],["Strong",c["STRONG"],COL[3]],["Very strong",c["VERY STRONG"],COL[4]]];
$("k").innerHTML=K.map(x=>`<div class="card kpi"><span>${x[0]}</span><b style="color:${x[2]}">${x[1]}</b></div>`).join("");
mk("c1","doughnut",Object.keys(c),Object.values(c),COL,{cutout:"62%",plugins:{legend:{position:"bottom",labels:{color:"#e8ecff"}}}});
mk("c2","bar",["0-9","10-19","20-29","30-39","40-49","50-59","60-69","70-79","80-89","90-100"],s.score_hist,"#8b9bff");
const w=Object.entries(s.weaknesses);mk("c3","bar",w.map(x=>x[0]),w.map(x=>x[1]),"#2ee6a6",{indexAxis:"y",plugins:{legend:{display:false}},scales:{x:{ticks:{color:"#c9d3f5"}},y:{ticks:{color:"#c9d3f5"}}}});
mk("c4","bar",Object.keys(s.lengths),Object.values(s.lengths),"#4cc9f0");
mk("c5","doughnut",Object.keys(s.patterns),Object.values(s.patterns),["#8b9bff","#ff9f43","#ffd166","#ff5d73","#2ee6a6"],{cutout:"55%",plugins:{legend:{position:"bottom",labels:{color:"#e8ecff"}}}})}
load();setInterval(load,5000);</script></body></html>"""

@app.get("/")
def home(): return Response(HOME.replace("__CSS__", CSS).replace("__NAV__", NAV % ("on", "")), mimetype="text/html")
@app.get("/dashboard")
def dash(): return Response(DASH.replace("__CSS__", CSS).replace("__NAV__", NAV % ("", "on")), mimetype="text/html")

# ---------------- TESTS (run once, printed as a table) ----------------
A = analyze_password
rand20 = generate_password(20)
T = [("T01", "Empty password", "''", lambda: A("")["score"] == 0),
     ("T02", "One character", "a", lambda: A("a")["classification"] == "VERY WEAK"),
     ("T03", "Short numeric", "1234", lambda: A("1234")["score"] <= 20),
     ("T04", "Common password", "123456", lambda: A("123456")["classification"] == "VERY WEAK"),
     ("T05", "Long repeated", "a*16", lambda: A("a" * 16)["classification"] in ("VERY WEAK", "WEAK")),
     ("T06", "Lowercase only", "kmqzvbwxplth", lambda: A("kmqzvbwxplth")["metrics"]["type_count"] == 1),
     ("T07", "Uppercase only", "KMQZVBWXPLTH", lambda: A("KMQZVBWXPLTH")["metrics"]["type_count"] == 1),
     ("T08", "Numbers only", "84629173", lambda: A("84629173")["metrics"]["type_count"] == 1),
     ("T09", "Symbols only", "!#%&*(", lambda: A("!#%&*(")["metrics"]["type_count"] == 1),
     ("T10", "Mixed characters", "Tq7#mZ", lambda: A("Tq7#mZ")["metrics"]["type_count"] == 4),
     ("T11", "Sequential numbers", "xx1234yy", lambda: "ascending" in detect_sequences("xx1234yy")),
     ("T12", "Reverse numeric", "9876", lambda: "descending" in detect_sequences("9876")),
     ("T13", "Sequential letters", "abcd", lambda: bool(detect_sequences("abcd"))),
     ("T14", "Keyboard sequence", "qwerty", lambda: bool(detect_keyboard_patterns("qwerty"))),
     ("T15", "Repeated characters", "1111", lambda: "repeated character" in detect_repetition("1111")),
     ("T16", "Repeated substring", "abcabcabc", lambda: "repeated substring" in detect_repetition("abcabcabc")),
     ("T17", "Word + number", "welcome123", lambda: any(f["type"] == "structure" for f in A("welcome123")["findings"])),
     ("T18", "Word + year", "admin2026", lambda: A("admin2026")["score"] <= 40),
     ("T19", "Personal name", "Rahul@123", lambda: any(f["type"] == "personal" for f in A("Rahul@123", {"name": "Rahul"})["findings"])),
     ("T20", "Birth year", "xk2004!!Qz", lambda: any(f["type"] == "personal" for f in A("xk2004!!Qz", {"year": "2004"})["findings"])),
     ("T21", "Passphrase", "velvet-galaxy-harbor-orchid-drift", lambda: A("velvet-galaxy-harbor-orchid-drift")["score"] >= 61),
     ("T22", "Unicode", "pässwörd-日本語-Ω9", lambda: isinstance(A("pässwörd-日本語-Ω9")["score"], int)),
     ("T23", "Spaces allowed", "blue tiger jumps high", lambda: A("blue tiger jumps high")["metrics"]["length"] == 21),
     ("T24", "Max length 128 accepted / 129 rejected", "x*128", lambda: app.test_client().post("/api/analyze", json={"password": "x" * 129}).status_code == 400),
     ("T25", "Score boundaries", "20/21/40/41/60/61/80/81", lambda: [classify(x) for x in (20, 21, 40, 41, 60, 61, 80, 81)] == ["VERY WEAK", "WEAK", "WEAK", "MODERATE", "MODERATE", "STRONG", "STRONG", "VERY STRONG"]),
     ("T26", "Suggestions generated", "qwerty2026!", lambda: len(A("qwerty2026!")["suggestions"]) >= 3),
     ("T27", "Secure generation", "len 24", lambda: len(generate_password(24)) == 24 and generate_password(20) != generate_password(20)),
     ("T28", "Random 20 chars strong", "<runtime>", lambda: A(rand20)["score"] >= 61),
     ("T29", "Hash/verify demo", "synthetic", lambda: (lambda h: verify_password("Demo-Only-Pw", h) and not verify_password("wrong", h))(hash_password("Demo-Only-Pw"))),
     ]
cl = app.test_client(); MARK = "Zq9-Marker-Pw-7731!x"
resp = cl.post("/api/analyze", json={"password": MARK, "record": True})
dump = "\n".join(db().iterdump()); cols = [r[1] for r in db().execute("PRAGMA table_info(analyses)")]
T += [("T30", "Password not in API response", "marker", lambda: MARK not in resp.get_data(as_text=True)),
      ("T31", "Password not in DB dump", "marker", lambda: MARK not in dump),
      ("T32", "No 'password' column in schema", "schema", lambda: "password" not in cols),
      ("T33", "Analytics stats metadata only", "stats", lambda: MARK not in json.dumps(stats())),
      ("T34", "Password not in URL/query handling", "GET ?password=", lambda: cl.get("/api/dashboard/stats?password=" + MARK).status_code == 200 and MARK not in cl.get("/api/dashboard/stats").get_data(as_text=True))]
print(f"{'ID':<5}{'Scenario':<40}{'Input':<36}Result"); ok = 0
for i, s, inp, f in T:
    try: r = bool(f())
    except Exception: r = False
    ok += r; print(f"{i:<5}{s:<40}{inp[:34]:<36}{'PASS' if r else 'FAIL'}")
print(f"\n{ok}/{len(T)} tests passed\n")
print("Safe demos:", {d: (A(d)["score"], A(d)["classification"]) for d in ["123456", "Password123!", "aaaaaaaaaaaaaaaa", "qwerty2026!"]}, "| random20 ->", A(rand20)["classification"])
print("Hash demo:", hash_password("Demo-Only-Pw")[:60] + "…\n")

# ---------------- LAUNCH + PUBLIC LINKS ----------------
PORT = 5055
threading.Thread(target=lambda: app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False), daemon=True).start(); time.sleep(1.5)

def start_tunnel(port):
    try:
        subprocess.run("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared", shell=True, check=True)
        p = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"], stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
        t0 = time.time()
        while time.time() - t0 < 45:
            m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", p.stderr.readline())
            if m:
                threading.Thread(target=lambda: [_ for _ in p.stderr], daemon=True).start(); return m.group(0)
    except Exception: pass
    try:
        from google.colab.output import eval_js
        return eval_js(f"google.colab.kernel.proxyPort({port})").rstrip("/")   # works only in your own browser
    except Exception: return None

base = start_tunnel(PORT)
if base:
    display(HTML(f"""<div style="font-family:Inter,sans-serif;background:#0a0f24;color:#f1f4ff;padding:20px;border-radius:14px;border:1px solid #2a3670">
    <b style="font-size:18px">🛡️ Password Strength Analyzer is live</b><br><br>
    🔎 Analyzer: <a style="color:#2ee6a6" href="{base}/" target="_blank">{base}/</a><br>
    📊 Dashboard: <a style="color:#4cc9f0" href="{base}/dashboard" target="_blank">{base}/dashboard</a><br><br>
    <span style="color:#aab5de">Public link: type ONLY synthetic demo passwords. Link stops working when the Colab runtime stops.</span></div>"""))
else:
    print("Could not create a public link. Local server is running on port", PORT)

ID   Scenario                                Input                               Result
T01  Empty password                          ''                                  PASS
T02  One character                           a                                   FAIL
T03  Short numeric                           1234                                FAIL
T04  Common password                         123456                              PASS
T05  Long repeated                           a*16                                PASS
T06  Lowercase only                          kmqzvbwxplth                        PASS
T07  Uppercase only                          KMQZVBWXPLTH                        PASS
T08  Numbers only                            84629173                            PASS
T09  Symbols only                            !#%&*(                              PASS
T10  Mixed characters                        Tq7#mZ                              PASS
T11  Sequential numbers                      xx1234y